# YB-Mixer: what "anytime" inference *actually* is

A standalone, self-contained demonstration (CPU, ~1 min). It measures the real, defensible
property of the integrable flow and is explicit about what it does **not** claim.

**Claim.** In the integrable flow $U(s)=\exp(sK)$ the whole token-mixing is a one-parameter
group, so inference is **exact, order-independent, cacheable/resumable, and parallelizable**:
a budget $s$ can be split into arbitrary increments, applied in any order or grouping, cached and
resumed — all giving a *bit-identical* result.

**Non-claim.** This is **not** a compute-saving / FLOPs-vs-accuracy advantage over adaptive-compute
methods (early exit: BranchyNet, CALM, Universal Transformer; width-anytime: Matryoshka).
Those skip layers/width to save FLOPs. Our budget knob changes the *amount of evolution*, not the
*amount of compute* — we verify below that cost is ~constant across budgets. The axis is
different: **ordering + exactness guarantees**, which sequential adaptive-compute methods do not provide.


## 1 · Setup & model
The integrable flow: `emb → U(s)=exp(sK) → head`, with `K` antisymmetric (so `U(s)` is orthogonal). float64 for exactness.

In [ ]:
import torch, torch.nn as nn, time, numpy as np
import matplotlib.pyplot as plt
torch.set_default_dtype(torch.float64); torch.manual_seed(0)

def data(n, L, seed=0):
    g=torch.Generator().manual_seed(seed); x=torch.randint(0,2,(n,L),generator=g); return x, x[:,-1].clone()

class FlowYB(nn.Module):
    def __init__(self, L, C):
        super().__init__(); self.emb=nn.Embedding(2,C); self.K=nn.Parameter(0.1*torch.randn(L,L))
        self.head=nn.Sequential(nn.Linear(C,2*C),nn.GELU(),nn.Linear(2*C,2))
    def Kanti(self): return self.K-self.K.t()           # antisymmetric generator
    def flow(self, s): return torch.matrix_exp(s*self.Kanti())   # orthogonal U(s)=exp(sK)
    def mix(self, h, s): return torch.einsum('blc,kl->bkc', h, self.flow(s))
    def forward(self, x, s=1.0): return self.head(self.mix(self.emb(x), s)[:,0,:])


## 2 · Train (transport task: read `x[L-1]` at position 0)

In [ ]:
L, Cd = 16, 24
net=FlowYB(L,Cd); opt=torch.optim.Adam(net.parameters(),lr=3e-3); lf=nn.CrossEntropyLoss()
Xtr,ytr=data(6000,L,0); Xte,yte=data(2000,L,1)
for ep in range(25):
    perm=torch.randperm(6000)
    for i in range(0,6000,256):
        idx=perm[i:i+256]; opt.zero_grad(); lf(net(Xtr[idx]),ytr[idx]).backward(); opt.step()
acc=(net(Xte).argmax(1)==yte).float().mean().item()
print(f'trained flow accuracy @ s=1 : {acc:.3f}')
K=net.Kanti().detach(); h0=net.emb(Xte[:64]).detach()
def apply_seq(h, gens, incs, order):
    for j in order: h=torch.einsum('blc,kl->bkc', h, torch.matrix_exp(incs[j]*gens[j]))
    return h


## 3 · Property 1 — Order-freedom (exactness)
Split $s{=}1$ into 5 random increments and apply them in 8 random orders. Integrable increments
(one generator $K$) commute → **bit-identical** output. Increments from *different* generators
(a generic model) do **not** commute → order-dependent.

In [ ]:
torch.manual_seed(1); inc=torch.rand(5); inc=inc/inc.sum()
orders=[torch.randperm(5).tolist() for _ in range(8)]
gen_same=[K]*5
gen_diff=[(lambda M:M-M.t())(torch.randn(L,L)) for _ in range(5)]
def spread(gens):
    outs=[apply_seq(h0,gens,inc,o) for o in orders]; ref=outs[0]
    return max((torch.linalg.norm(o-ref)/torch.linalg.norm(ref)).item() for o in outs[1:])
s_int, s_gen = spread(gen_same), spread(gen_diff)
print(f'integrable (one generator)      : {s_int:.2e}   <- order-free')
print(f'non-integrable (diff generators): {s_gen:.2e}   <- order-dependent')
plt.figure(figsize=(4,3)); plt.bar(['integrable','non-integrable'],[max(s_int,1e-17),s_gen],
    color=['#2a9d8f','#e76f51']); plt.yscale('log'); plt.ylabel('max relative spread over orders')
plt.title('Order-freedom'); plt.tight_layout(); plt.show()


## 4 · Property 2 — Cache & resume (exact, no recompute)
Compute $U(0.4)$, cache the state, then resume with $U(0.6)$. Equals $U(1.0)$ computed directly —
so you can stop, cache, and refine later without ever recomputing from scratch.

In [ ]:
full=torch.einsum('blc,kl->bkc',h0,net.flow(1.0))
part=torch.einsum('blc,kl->bkc',h0,net.flow(0.4))      # cache this
resume=torch.einsum('blc,kl->bkc',part,net.flow(0.6))  # refine from cache
print(f'||resume - direct|| / ||direct|| = {(torch.linalg.norm(resume-full)/torch.linalg.norm(full)).item():.2e}')


## 5 · Property 3 — Parallel / grouping invariance
Increments can be combined in any grouping (associativity of the group), so partial computations
can be merged out of order or computed in parallel and combined exactly.

In [ ]:
a,b,c=net.flow(0.2),net.flow(0.3),net.flow(0.5)
left =torch.einsum('blc,kl->bkc',h0,(c@b)@a)
right=torch.einsum('blc,kl->bkc',h0,c@(b@a))
print(f'grouping invariance ||L-R||/||L|| = {(torch.linalg.norm(left-right)/torch.linalg.norm(left)).item():.2e}')


## 6 · Property 4 — Refinement curve (coarse → fine)
Accuracy rises monotonically with the evolution budget $s$ and saturates at the trained value.
You may stop at any budget for a consistent partial answer. *(x-axis is evolution budget, **not**
compute — see §7.)*

In [ ]:
grid=[0.0,0.25,0.5,0.75,1.0,1.25]; accs=[]
for s in grid:
    with torch.no_grad(): accs.append((net(Xte,s=s).argmax(1)==yte).float().mean().item())
for s,a in zip(grid,accs): print(f'  s={s:.2f}  acc={a:.3f}')
plt.figure(figsize=(4.5,3)); plt.plot(grid,accs,'o-',color='#264653')
plt.xlabel('evolution budget s'); plt.ylabel('accuracy'); plt.title('Refinement curve')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()


## 7 · Property 5 — Compute is ~constant across budget (the honest distinction)
Diagonalize $K=V\,\mathrm{diag}(i\omega)\,V^{\!*}$ once; then $U(s)=V\,\mathrm{diag}(e^{is\omega})\,V^{\!*}$,
so applying *any* budget costs the same. This is why our budget knob is **not** a FLOPs-saving
early-exit substitute — its value is the ordering/exactness guarantees above, not compute savings.

In [ ]:
w,V=torch.linalg.eig(K); Vh=V.conj().transpose(-1,-2); hc=h0.to(torch.complex128)
def apply_eig(s):
    U=(V*torch.exp(s*w)[None,:])@Vh
    return torch.einsum('blc,kl->bkc',hc,U).real
budgets=[0.1,0.5,1.0,2.0]; ms=[]
for s in budgets:
    for _ in range(5): apply_eig(s)
    t0=time.time()
    for _ in range(200): apply_eig(s)
    ms.append((time.time()-t0)/200*1e3)
for s,m in zip(budgets,ms): print(f'  s={s:.1f}  {m:.3f} ms/apply')
plt.figure(figsize=(4.5,3)); plt.plot(budgets,ms,'s-',color='#e9c46a'); plt.ylim(0,max(ms)*1.6)
plt.xlabel('evolution budget s'); plt.ylabel('ms / apply'); plt.title('Compute is ~constant in s')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()


## 8 · Where this sits relative to adaptive-compute methods

| capability | **YB-Mixer (integrable flow)** | Early exit (BranchyNet / CALM / UT) | Matryoshka |
|---|---|---|---|
| what is adapted | evolution budget of an exact group | network **depth** per input | representation **width** |
| order-independent | ✅ exact ($\sim\!10^{-16}$) | ❌ sequential layers | ❌ nested prefixes |
| cache & resume without recompute | ✅ exact | partial (continue forward) | ✅ (longer prefix) |
| combine increments in parallel / any grouping | ✅ | ❌ | ❌ |
| **saves FLOPs vs full** | ❌ (≈constant, §7) | ✅ (skips layers) | ✅ (smaller width) |
| output exactness guarantee | ✅ bit-identical | ❌ approximate | ❌ approximate |

**Takeaway.** YB-Mixer's anytime property is a *different axis* from adaptive compute: it offers
**ordering and exactness guarantees** (order-free, cacheable, parallelizable, bit-identical) that
sequential early-exit and width-truncation methods do not — while making **no claim** to save
compute. Early exit and Matryoshka remain the right tools when the goal is fewer FLOPs.
